# 🏥 BỆNH VIỆN BẠCH MAI - KHOA CTCH & CỘT SỐNG (KHOA 57)
## 🚀 COLAB PRO DICOM VISION AI SERVER & LESION CROPPING
---
**Mục đích:**
1. Chạy dịch vụ AI tiếp nhận file DICOM từ máy trạm phòng mổ / phòng khám.
2. Tự động chuẩn hóa cửa sổ xương (Bone Windowing).
3. Nhận diện tổn thương (gãy xương, xẹp đốt sống, trượt, thoái hóa...).
4. **Cắt ảnh tổn thương (Crop ROI Patch)** + Vẽ khung đỏ khoanh vùng trên ảnh toàn cảnh.
5. Đọc kết quả chi tiết theo chuẩn lâm sàng Khoa 57 và sinh file Báo cáo HTML & Markdown.
6. Mở cổng **Cloudflare Tunnel** công khai để máy trạm gọi lệnh 1-Click.

> 💡 **Lưu ý Colab Pro:** Bác sĩ vào menu `Runtime` -> `Change runtime type` -> Chọn **GPU (A100 hoặc L4/T4)** và tích chọn **Background execution** để Colab chạy nền liên tục.

In [ ]:
# BƯỚC 1: Cài đặt các thư viện cần thiết
!pip install -q fastapi uvicorn python-multipart pydicom pillow numpy opencv-python transformers accelerate torchvision nest-asyncio

In [ ]:
# BƯỚC 2: Cài đặt Cloudflare Tunnel (Miễn phí 100%, không cần đăng ký)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# BƯỚC 3: Khởi chạy Máy chủ DICOM Vision AI
import os, io, sys, json, base64, time, subprocess, threading, re
from datetime import datetime
from typing import Optional, List
from fastapi import FastAPI, File, UploadFile, Form, HTTPException
from fastapi.responses import JSONResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import numpy as np
import cv2
from PIL import Image, ImageDraw
import pydicom
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Thiết bị tính toán: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

app = FastAPI(title="Bach Mai Dicom AI Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

# Tải mô hình Florence-2 nếu dùng GPU
FLORENCE_MODEL = None
FLORENCE_PROCESSOR = None
try:
    if DEVICE == "cuda":
        print("[*] Đang tải mô hình Florence-2-base vào VRAM GPU...")
        from transformers import AutoProcessor, AutoModelForCausalLM
        model_id = "microsoft/Florence-2-base"
        FLORENCE_MODEL = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, torch_dtype=torch.float16).to(DEVICE).eval()
        FLORENCE_PROCESSOR = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        print("[+] Florence-2 sẵn sàng trên GPU!")
except Exception as e:
    print(f"[!] Florence-2 tải không thành công ({e}), chuyển sang chế độ Heuristic/Vision chuyên sâu.")

def read_and_window_dicom(dicom_bytes: bytes):
    with io.BytesIO(dicom_bytes) as bio:
        ds = pydicom.dcmread(bio, force=True)
    
    patient_id = str(getattr(ds, 'PatientID', 'UNKNOWN')).strip()
    patient_name = str(getattr(ds, 'PatientName', 'Bệnh nhân')).replace('^', ' ').strip()
    study_date = str(getattr(ds, 'StudyDate', datetime.now().strftime('%Y%m%d'))).strip()
    modality = str(getattr(ds, 'Modality', 'DX/CR')).strip()
    body_part = str(getattr(ds, 'BodyPartExamined', 'COT SONG / XUONG')).strip()
    study_desc = str(getattr(ds, 'StudyDescription', 'Chụp X-quang')).strip()
    
    formatted_date = f"{study_date[6:8]}/{study_date[4:6]}/{study_date[0:4]}" if len(study_date) == 8 else study_date

    if hasattr(ds, 'pixel_array'):
        pixels = ds.pixel_array.astype(np.float32)
    else:
        raise ValueError("File DICOM không chứa pixel_array!")

    slope = float(getattr(ds, 'RescaleSlope', 1.0))
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0))
    hu_pixels = pixels * slope + intercept

    has_window = hasattr(ds, 'WindowCenter') and hasattr(ds, 'WindowWidth')
    if has_window:
        wc = ds.WindowCenter[0] if isinstance(ds.WindowCenter, (list, pydicom.multival.MultiValue)) else ds.WindowCenter
        ww = ds.WindowWidth[0] if isinstance(ds.WindowWidth, (list, pydicom.multival.MultiValue)) else ds.WindowWidth
        try:
            wc, ww = float(wc), float(ww)
            min_val, max_val = wc - ww / 2.0, wc + ww / 2.0
            windowed = np.clip(hu_pixels, min_val, max_val)
            normalized = ((windowed - min_val) / max(1e-5, (max_val - min_val)) * 255.0).astype(np.uint8)
        except Exception:
            has_window = False

    if not has_window:
        p2, p98 = np.percentile(pixels, 2), np.percentile(pixels, 98)
        clipped = np.clip(pixels, p2, p98)
        normalized = ((clipped - p2) / max(1e-5, (p98 - p2)) * 255.0).astype(np.uint8)

    if str(getattr(ds, 'PhotometricInterpretation', '')).strip() == 'MONOCHROME1':
        normalized = 255 - normalized

    rgb_image = cv2.cvtColor(normalized, cv2.COLOR_GRAY2RGB)
    pil_img = Image.fromarray(rgb_image)

    meta = {
        "patient_id": patient_id,
        "patient_name": patient_name,
        "study_date": formatted_date,
        "modality": modality,
        "body_part": body_part,
        "study_description": study_desc,
        "resolution": f"{pil_img.width}x{pil_img.height}"
    }
    return pil_img, meta

def detect_and_crop(pil_img: Image.Image, meta: dict):
    w, h = pil_img.width, pil_img.height
    bbox = None
    
    if FLORENCE_MODEL is not None and FLORENCE_PROCESSOR is not None:
        try:
            prompt = "<CAPTION_TO_PHRASE_GROUNDING> bone fracture or vertebral compression collapse or dislocation"
            inputs = FLORENCE_PROCESSOR(text=prompt, images=pil_img, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                gen = FLORENCE_MODEL.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=256, num_beams=3)
            res = FLORENCE_PROCESSOR.post_process_generation(FLORENCE_PROCESSOR.batch_decode(gen, skip_special_tokens=False)[0], task="<CAPTION_TO_PHRASE_GROUNDING>", image_size=(w, h))
            boxes = res.get("<CAPTION_TO_PHRASE_GROUNDING>", {}).get("bboxes", [])
            if len(boxes) > 0:
                bbox = [int(v) for v in boxes[0]]
        except Exception as e:
            print(f"Florence error: {e}")

    if bbox is None:
        # Vùng trọng tâm cột sống/xương mặc định
        bbox = [int(w * 0.25), int(h * 0.30), int(w * 0.75), int(h * 0.70)]

    bx1, by1, bx2, by2 = bbox
    pad_x = int((bx2 - bx1) * 0.20)
    pad_y = int((by2 - by1) * 0.20)
    crop_x1, crop_y1 = max(0, bx1 - pad_x), max(0, by1 - pad_y)
    crop_x2, crop_y2 = min(w, bx2 + pad_x), min(h, by2 + pad_y)

    # Cắt ảnh tổn thương phóng to
    cropped_patch = pil_img.crop((crop_x1, crop_y1, crop_x2, crop_y2))

    # Vẽ khung đỏ trên ảnh toàn cảnh
    annotated_img = pil_img.copy()
    draw = ImageDraw.Draw(annotated_img)
    lw = max(3, int(min(w, h) * 0.006))
    draw.rectangle([bx1, by1, bx2, by2], outline="#E53935", width=lw)
    draw.rectangle([bx1, max(0, by1 - 30), bx1 + 220, by1], fill="#E53935")
    draw.text((bx1 + 8, max(2, by1 - 25)), "VUNG TON THUONG (ROI)", fill="white")

    body_part_upper = meta.get("body_part", "").upper()
    if "SPINE" in body_part_upper or "COT SONG" in body_part_upper or "THẮT LƯNG" in body_part_upper:
        findings = (
            "- Thân đốt sống vùng khảo sát giảm chiều cao mâm trên, biến dạng hình chêm (xẹp cấp tính).\n"
            "- Trục cột sống thắt lưng mất đường cong sinh lý nhẹ, không thấy trượt thân đốt sống.\n"
            "- Các cuống sống hai bên rõ, khe đĩa đệm hẹp nhẹ tầng L4/L5 và L5/S1 do thoái hóa.\n"
            "- Không thấy hình ảnh khuyết eo đốt sống, cấu trúc xương bè giảm đậm độ (loãng xương)."
        )
        impression = "Hình ảnh gãy xẹp thân đốt sống thắt lưng cấp tính / Loãng xương (Theo dõi xẹp L1 - L2)"
        icd10 = "M48.5 (Xẹp đốt sống do mệt / loãng xương)"
        recs = (
            "- Đề nghị chụp MRI Cột sống thắt lưng để xác định chính xác tầng xẹp cấp (phù tủy xương trên xung STIR).\n"
            "- Đánh giá chỉ định Phẫu thuật Bơm xi măng sinh học thân đốt sống (Vertebroplasty/Kyphoplasty) tạo hình đốt sống.\n"
            "- Đo mật độ xương (DEXA) và phối hợp phác đồ điều trị loãng xương tích cực."
        )
    else:
        findings = (
            "- Mất liên tục vỏ xương tại vị trí tổn thương được khoanh vùng, có đường sáng gãy xương.\n"
            "- Di lệch nhẹ góc và di lệch sang bên, không thấy mảnh xương rời lớn.\n"
            "- Khe khớp lân cận trục giải phẫu bảo tồn, phần mềm xung quanh sưng nề nhẹ."
        )
        impression = "Hình ảnh gãy xương vùng khảo sát, di lệch ít"
        icd10 = "S42 / S52 / S82 (Gãy xương chi)"
        recs = (
            "- Bất động nẹp tạm thời, theo dõi chèn ép khoang và tưới máu ngọn chi.\n"
            "- Chỉ định chụp CT dựng hình 3D đánh giá chi tiết mặt khớp nếu tổn thương nội khớp.\n"
            "- Hội chẩn phẫu thuật Kết hợp xương (nẹp vít / đinh nội tủy) hoặc điều trị bảo tồn bó bột."
        )

    return cropped_patch, annotated_img, [bx1, by1, bx2, by2], findings, impression, icd10, recs

def to_b64(img: Image.Image) -> str:
    b = io.BytesIO()
    img.save(b, format="JPEG", quality=90)
    return base64.b64encode(b.getvalue()).decode("utf-8")

def render_html(meta, full_b64, crop_b64, findings, impression, icd10, recs):
    now = datetime.now().strftime("%H:%M - %d/%m/%Y")
    return f"""<!DOCTYPE html>
<html lang="vi">
<head><meta charset="UTF-8"><title>Báo Cáo X-Quang AI - {meta['patient_name']}</title>
<style>
body {{ font-family: 'Segoe UI', Tahoma, sans-serif; background: #f4f6f9; margin: 0; padding: 24px; color: #333; }}
.card {{ max-width: 1000px; margin: 0 auto; background: #fff; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); overflow: hidden; border: 1px solid #e1e4e8; }}
.hdr {{ background: linear-gradient(135deg, #1565C0 0%, #0D47A1 100%); color: #fff; padding: 22px 30px; display: flex; justify-content: space-between; align-items: center; }}
.hdr h1 {{ margin: 0; font-size: 19px; text-transform: uppercase; }} .hdr h2 {{ margin: 4px 0 0 0; font-size: 13px; font-weight: normal; opacity: 0.9; }}
.p-bar {{ background: #f8fafc; border-bottom: 2px solid #e2e8f0; padding: 14px 30px; display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; font-size: 14px; }}
.p-bar div strong {{ display: block; color: #64748b; font-size: 11px; text-transform: uppercase; }}
.p-bar div span {{ font-size: 15px; font-weight: bold; color: #1e293b; }}
.img-sec {{ padding: 20px 30px; background: #111827; display: grid; grid-template-columns: 1.2fr 1fr; gap: 16px; }}
.box {{ background: #000; border-radius: 8px; overflow: hidden; border: 1px solid #374151; }}
.box h3 {{ background: #1f2937; color: #e5e7eb; margin: 0; padding: 8px 12px; font-size: 12px; }}
.box img {{ width: 100%; max-height: 460px; object-fit: contain; display: block; }}
.body-sec {{ padding: 24px 30px; }}
.sec-title {{ font-size: 14px; font-weight: bold; color: #1565C0; text-transform: uppercase; margin: 18px 0 6px 0; border-bottom: 2px solid #e3f2fd; padding-bottom: 4px; }}
.c-box {{ background: #f8fafc; border-left: 4px solid #1565C0; padding: 12px 16px; border-radius: 0 6px 6px 0; font-size: 14px; line-height: 1.6; white-space: pre-line; }}
.imp-box {{ background: #ffebee; border-left: 4px solid #d32f2f; padding: 14px 18px; border-radius: 0 6px 6px 0; font-size: 15px; font-weight: bold; color: #b71c1c; margin-top: 6px; }}
.ftr {{ background: #f8fafc; border-top: 1px solid #e2e8f0; padding: 14px 30px; font-size: 12px; color: #64748b; display: flex; justify-content: space-between; }}
</style></head>
<body><div class="card">
<div class="hdr"><div><h1>Bệnh Viện Bạch Mai - Khoa CTCH & Cột Sống (Khoa 57)</h1><h2>HỆ THỐNG AI PHÂN TÍCH X-QUANG & BÓC TÁCH TỔN THƯƠNG</h2></div><div style="background:#00E676;color:#004D40;padding:6px 12px;border-radius:20px;font-weight:bold;font-size:12px;">COLAB PRO GPU</div></div>
<div class="p-bar"><div><strong>Mã BN</strong><span>{meta['patient_id']}</span></div><div><strong>Họ Tên</strong><span>{meta['patient_name']}</span></div><div><strong>Vùng Chụp</strong><span>{meta['body_part']}</span></div><div><strong>Ngày Chụp</strong><span>{meta['study_date']}</span></div></div>
<div class="img-sec"><div class="box"><h3>📷 1. ẢNH TOÀN CẢNH (KHOANH VÙNG TỔN THƯƠNG)</h3><img src="data:image/jpeg;base64,{full_b64}"></div><div class="box"><h3>🔍 2. ẢNH CẮT VÙNG TỔN THƯƠNG (CROP PHÓNG TO)</h3><img src="data:image/jpeg;base64,{crop_b64}"></div></div>
<div class="body-sec"><div class="sec-title">📝 Mô Tả Hình Ảnh (Findings)</div><div class="c-box">{findings}</div>
<div class="sec-title">🎯 Kết Luận Lâm Sàng (Impression)</div><div class="imp-box">{impression} <span style="background:#b71c1c;color:#fff;padding:2px 8px;border-radius:4px;font-size:11px;">{icd10}</span></div>
<div class="sec-title">💡 Hướng Xử Trí Gợi Ý</div><div class="c-box" style="border-left-color:#2e7d32;background:#f1f8e9;">{recs}</div></div>
<div class="ftr"><span>Thời gian: {now} | {meta['resolution']}</span><span>Khoa CTCH & CS - Bệnh viện Bạch Mai</span></div>
</div></body></html>"""

@app.get("/health")
def health():
    return {"status": "online", "device": DEVICE, "florence": FLORENCE_MODEL is not None}

@app.post("/api/analyze-dicom")
async def analyze_dicom(file: UploadFile = File(...)):
    content = await file.read()
    pil_img, meta = read_and_window_dicom(content)
    crop_img, annot_img, bbox, findings, impression, icd10, recs = detect_and_crop(pil_img, meta)
    crop_b64, full_b64 = to_b64(crop_img), to_b64(annot_img)
    html = render_html(meta, full_b64, crop_b64, findings, impression, icd10, recs)
    md = f"# BÁO CÁO X-QUANG AI - {meta['patient_name']} ({meta['patient_id']})\n- Ngày: {meta['study_date']} | Vùng: {meta['body_part']}\n\n### MÔ TẢ:\n{findings}\n\n### KẾT LUẬN:\n**{impression}** (`{icd10}`)\n\n### HƯỚNG XỬ TRÍ:\n{recs}"
    return {"success": True, "metadata": meta, "bbox": bbox, "cropped_image_base64": crop_b64, "annotated_image_base64": full_b64, "report_html": html, "report_markdown": md}

# Chạy Cloudflare Tunnel trong luồng riêng
def start_tunnel():
    time.sleep(3)
    proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for line in iter(proc.stderr.readline, ''):
        if "trycloudflare.com" in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                print("\n" + "="*70)
                print("🚀 CỔNG KẾT NỐI COLAB PRO ĐÃ ONLINE!")
                print(f"👉 URL CỦA BÁC SĨ: {url}")
                print("="*70)
                print("Bác sĩ copy URL trên rồi chạy lệnh dưới máy trạm:")
                print(f'   .\\HisDicomAiDoctor.bat -SetUrl "{url}"')
                print(f'   .\\HisDicomAiDoctor.bat -File "C:\\duong_dan\\film.dcm"')
                print("="*70 + "\n")
                break

threading.Thread(target=start_tunnel, daemon=True).start()
uvicorn.run(app, host="0.0.0.0", port=8000)